<a href="https://colab.research.google.com/github/ibrahimymhafez/flyrank-ibrahim/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ibrahimymhafez/flyrank-ibrahim/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

**Finding 1: Finding #4 states that refreshing mature pages (365+ days old) produces a 3.2x health boost and a 57x impression boost.**

Methodology Question: Does the validation design control for selection bias? If editors naturally choose to refresh their historically best-performing or most strategic pages, the massive 57x impression boost might be skewed by the inherent quality of the chosen pages rather than just the act of refreshing them.

**Finding 2: Myth #3 claims that content with 2-3 optimization flags actually scores better (Health 42-47) than content with zero flags (Health 36) because flags require baseline visibility to trigger.**

Methodology Question: Where does the "Health Score" label come from in relation to these flags? Since Health Score is explicitly constructed using impressions (30 pts), and flags require impressions to trigger, is the validation design simply capturing an artificial loop where both metrics share the same underlying mathematical input?

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [ ]:
import duckdb
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score
from google.colab import userdata

# Connect to DuckDB via Hugging Face token
hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

# 1. BEFORE: Random Split (Leaky across time)
# Fetching a mixed batch of data across time, but using honest features
query_leaky = """
    SELECT
        gsc_impressions as impressions,
        gsc_avg_position as position,
        CAST((gsc_clicks = 0 AND gsc_impressions > 0) AS INTEGER) as needs_refresh
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/*/*.parquet'
    WHERE gsc_impressions IS NOT NULL LIMIT 20000
"""
df_leaky = con.sql(query_leaky).df().fillna(0)

# We use ONLY non-leaky features here
X_leak = df_leaky[['impressions', 'position']]
y_leak = df_leaky['needs_refresh']
X_train_rand, X_test_rand, y_train_rand, y_test_rand = train_test_split(X_leak, y_leak, test_size=0.2, random_state=42)

rf_random = RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42)
rf_random.fit(X_train_rand, y_train_rand)
leaky_precision = precision_score(y_test_rand, rf_random.predict(X_test_rand), zero_division=0)

# 2. AFTER: Honest Time-Aware Split (March -> April)
df_train = con.sql("""
    SELECT
        gsc_impressions as impressions,
        gsc_avg_position as position,
        CAST((gsc_clicks = 0 AND gsc_impressions > 0) AS INTEGER) as needs_refresh
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    WHERE gsc_impressions IS NOT NULL LIMIT 16000
""").df().fillna(0)

df_test = con.sql("""
    SELECT
        gsc_impressions as impressions,
        gsc_avg_position as position,
        CAST((gsc_clicks = 0 AND gsc_impressions > 0) AS INTEGER) as needs_refresh
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet'
    WHERE gsc_impressions IS NOT NULL LIMIT 4000
""").df().fillna(0)

rf_honest = RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42)
rf_honest.fit(df_train[['impressions', 'position']], df_train['needs_refresh'])
honest_precision = precision_score(df_test['needs_refresh'], rf_honest.predict(df_test[['impressions', 'position']]), zero_division=0)

print(f"BEFORE (Random Split Precision): {leaky_precision:.2%}")
print(f"AFTER (Time-Aware Split Precision): {honest_precision:.2%}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

BEFORE (Random Split Precision): 92.04%
AFTER (Time-Aware Split Precision): 95.67%


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Programmatically hunt for leakage via Pearson correlation, matching the paper's ML Appendix
print("--- Leakage Audit: Feature Correlation to Target ---")
correlations = df_train.corr()['needs_refresh'].sort_values(ascending=False)
display(correlations)

# Define strict leakage thresholds (ignoring the target itself which is 1.0)
leaky_features = correlations[(correlations > 0.8) & (correlations < 1.0)]

if not leaky_features.empty:
    print(f"🚨 WARNING: Potential leakage found. High correlation with label:\n{leaky_features}")
else:
    print("✅ CONFIRMED: No features show suspicious, leakage-level correlation with the label.")

--- Leakage Audit: Feature Correlation to Target ---


,needs_refresh
needs_refresh,1.000000
position,0.410896
impressions,0.007954


✅ CONFIRMED: No features show suspicious, leakage-level correlation with the label.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*


Original bold claim (Unsafe): "My Random Forest model perfectly predicts Google's algorithm, guaranteeing that if the editorial team rewrites these specific pages, we will recover all lost traffic and stop content churn."

Rewritten safe language (Public-Safe): "Based on measured historical data, the model identified pages exhibiting a directional decline in search engagement. These ranked outputs are designed to provide decision-support for the editorial team, helping them prioritize their content refresh queue based on observed performance patterns."

## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.